In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.data_loader import load_men_data
from src.feature_engineering import (
    compute_elo_ratings, parse_seeds, compute_season_stats,
    compute_massey_features, compute_conference_strength,
    build_team_features, build_matchup_matrix, elo_to_prob
)

In [2]:
data_dir = f'{project_root}/data'

men = load_men_data(data_dir)

In [3]:
men_elo = compute_elo_ratings(men['MComSsn'])
men_seeds = parse_seeds(men['MTrnySeeds'])
men_stats = compute_season_stats(men['MDetSsn'])
massey = compute_massey_features(men['MOrdinals'])
conf = compute_conference_strength(men['MConf'], men_elo)

men_features = build_team_features(men_elo, men_seeds, men_stats, massey, conf)
print(men_features.shape)
print(men_features.columns.tolist())

matchups = build_matchup_matrix(men['MDetTrny'], men_features)
# Or if you have compact tourney results loaded separately:
# matchups = build_matchup_matrix(men['MDetTrny'], men_features)
print(matchups.shape)
print(matchups.head())
print(matchups['Target'].mean())  # should be ~0.5 (no bias toward lower/higher IDs)
print(matchups.isnull().sum())

(14206, 25)
['Season', 'TeamID', 'Elo', 'SeedNum', 'eFG_off', 'TO_rate_off', 'OR_pct', 'FT_rate_off', 'eFG_def', 'TO_rate_def', 'DR_pct', 'FT_rate_def', 'Tempo', 'PPG', 'PPG_allowed', 'Win_pct', 'Last10_Win_pct', 'Massey_median', 'Massey_mean', 'Massey_min', 'Massey_std', 'Rank_RPI', 'Rank_SAG', 'Rank_POM', 'Conf_Elo_mean']
(1449, 27)
   Season  TeamA  TeamB    Elo_diff  SeedNum_diff  eFG_off_diff  \
0    2003   1411   1421   13.847729           0.0      0.013235   
1    2003   1112   1436  510.743224         -15.0      0.022900   
2    2003   1113   1272 -114.782588           3.0      0.018997   
3    2003   1141   1166  -48.630197           5.0      0.005381   
4    2003   1143   1301   30.641004          -1.0     -0.010092   

   TO_rate_off_diff  OR_pct_diff  FT_rate_off_diff  eFG_def_diff  ...  \
0         -0.012414     0.012949          0.042472     -0.024351  ...   
1         -0.019188     0.014010          0.037130     -0.020695  ...   
2          0.006264     0.031277         

In [4]:
from src.model import leave_one_season_out_cv_lean, train_final_model, print_feature_importance

oof_lean = leave_one_season_out_cv_lean(matchups)

model, feature_cols = train_final_model(matchups)
print_feature_importance(model)

Season 2003: Brier=0.20700  (n=64)
Season 2004: Brier=0.16992  (n=64)
Season 2005: Brier=0.19184  (n=64)
Season 2006: Brier=0.18420  (n=64)
Season 2007: Brier=0.16132  (n=64)
Season 2008: Brier=0.15630  (n=64)
Season 2009: Brier=0.17511  (n=64)
Season 2010: Brier=0.19089  (n=64)
Season 2011: Brier=0.22656  (n=67)
Season 2012: Brier=0.18045  (n=67)
Season 2013: Brier=0.23841  (n=67)
Season 2014: Brier=0.21385  (n=67)
Season 2015: Brier=0.16408  (n=67)
Season 2016: Brier=0.20001  (n=67)
Season 2017: Brier=0.19769  (n=67)
Season 2018: Brier=0.19604  (n=67)
Season 2019: Brier=0.15624  (n=67)
Season 2021: Brier=0.21430  (n=66)
Season 2022: Brier=0.22580  (n=67)
Season 2023: Brier=0.20008  (n=67)
Season 2024: Brier=0.20561  (n=67)
Season 2025: Brier=0.15362  (n=67)

Overall OOF Brier (lean): 0.19151
Top features by importance:
  Rank_POM_diff                  14.7696
  Massey_mean_diff               6.5929
  Elo_diff                       5.3087
  Massey_std_diff                4.6511
  Mass

In [7]:
from src.model import find_best_blend

# Compute Elo probabilities for the same games
elo_a = oof_lean.merge(men_elo, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'])['Elo']
elo_b = oof_lean.merge(men_elo, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'])['Elo']
elo_probs = elo_to_prob(elo_a.values, elo_b.values)

# Find best blend
best_w, best_brier = find_best_blend(oof_lean, elo_probs)

w=0.00 (pure Elo):    Brier=0.19250
w=0.05:               Brier=0.19168
w=0.10:               Brier=0.19094
w=0.15:               Brier=0.19028
w=0.20:               Brier=0.18971
w=0.25:               Brier=0.18921
w=0.30:               Brier=0.18880
w=0.35:               Brier=0.18847
w=0.40:               Brier=0.18822
w=0.45:               Brier=0.18804
w=0.50:               Brier=0.18795
w=0.55:               Brier=0.18795
w=0.60:               Brier=0.18802
w=0.65:               Brier=0.18817
w=0.70:               Brier=0.18841
w=0.75:               Brier=0.18872
w=0.80:               Brier=0.18912
w=0.85:               Brier=0.18959
w=0.90:               Brier=0.19015
w=0.95:               Brier=0.19079
w=1.00:               Brier=0.19151 <-- pure XGB

Best blend: w=0.55, Brier=0.18795
